# Component Explorer

Exercise **every component** with all valid configurations.  Organised by
pipeline stage: OCR → Refiners → Chunkers → Embedders.

> **Note:** This notebook calls every supported provider.  Make sure the
> corresponding API keys are set in your `.env` file.

In [ ]:
# ── Change these to use your own document and API keys ──
PDF_PATH = "../tests/fixtures/sample.pdf"
ENV_PATH = "../tests/.env"
# ────────────────────────────────────────────────────────

import os, logging
from dotenv import load_dotenv

load_dotenv(ENV_PATH)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
DATALAB_API_KEY = os.getenv("DATALAB_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
VOYAGE_API_KEY = os.getenv("VOYAGE_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

print("Setup complete ✓")

---
## 1 — OCR Providers

Run every valid OCR configuration on the sample PDF.

In [ ]:
from ragbandit.documents import MistralOCR, DatalabOCR

ocr_configs = [
    ("MistralOCR (mistral-ocr-2512)", MistralOCR(api_key=MISTRAL_API_KEY, model="mistral-ocr-2512")),
    ("MistralOCR (mistral-ocr-2505)", MistralOCR(api_key=MISTRAL_API_KEY, model="mistral-ocr-2505")),
    ("DatalabOCR (fast)",             DatalabOCR(api_key=DATALAB_API_KEY, mode="fast")),
    ("DatalabOCR (balanced)",         DatalabOCR(api_key=DATALAB_API_KEY, mode="balanced")),
    ("DatalabOCR (accurate)",         DatalabOCR(api_key=DATALAB_API_KEY, mode="accurate")),
]

ocr_results = {}
for name, ocr in ocr_configs:
    print(f"\n{'─' * 50}")
    print(f"  {name}")
    print(f"{'─' * 50}")
    result = ocr.process(PDF_PATH)
    ocr_results[name] = result
    print(f"  Pages  : {len(result.pages)}")
    print(f"  Model  : {result.model}")
    preview = result.pages[0].markdown[:150].replace('\n', ' ')
    print(f"  Page 0 : {preview}...")

---
## 2 — Refiners

Run each refiner on the first Mistral OCR result.

In [ ]:
from ragbandit.documents import ReferencesRefiner, FootnoteRefiner, TableOfContentsRefiner

# Use the first Mistral OCR result as input for refiners
base_ocr = ocr_results["MistralOCR (mistral-ocr-2512)"]

refiner_configs = [
    ("ReferencesRefiner",      ReferencesRefiner(api_key=MISTRAL_API_KEY)),
    ("FootnoteRefiner",        FootnoteRefiner(api_key=MISTRAL_API_KEY)),
    ("TableOfContentsRefiner", TableOfContentsRefiner(api_key=MISTRAL_API_KEY)),
]

refiner_results = {}
for name, refiner in refiner_configs:
    print(f"\n{'─' * 50}")
    print(f"  {name}")
    print(f"{'─' * 50}")
    rr = refiner.process(base_ocr)
    refiner_results[name] = rr
    print(f"  Pages          : {len(rr.pages)}")
    print(f"  Refiner        : {rr.component_name}")
    if rr.extracted_data:
        for key, val in rr.extracted_data.items():
            preview = str(val)[:120].replace('\n', ' ')
            print(f"  {key}: {preview}")

---
## 3 — Chunkers

Run every chunker configuration on a shared refined result and compare stats.

In [ ]:
from ragbandit.documents import (
    FixedSizeChunker,
    SentenceChunker,
    RecursiveMarkdownChunker,
    SemanticChunker,
)
from ragbandit.documents.refiners.base_refiner import BaseRefiner

# Convert OCR result to RefiningResult for chunkers
ref_result = BaseRefiner.ensure_refining_result(base_ocr)

chunker_configs = [
    ("FixedSize(1000/200)",          FixedSizeChunker(chunk_size=1000, overlap=200)),
    ("FixedSize(500/50)",            FixedSizeChunker(chunk_size=500, overlap=50)),
    ("Sentence(5/1, min=100)",       SentenceChunker(sentences_per_chunk=5, sentence_overlap=1, min_chunk_size=100, max_chunk_size=30000)),
    ("Sentence(3/0)",               SentenceChunker(sentences_per_chunk=3, sentence_overlap=0)),
    ("RecursiveMarkdown(1000/100)",  RecursiveMarkdownChunker(chunk_size=1000, overlap=100)),
    ("RecursiveMarkdown(500/50)",    RecursiveMarkdownChunker(chunk_size=500, overlap=50)),
    ("Semantic(min=500)",            SemanticChunker(api_key=MISTRAL_API_KEY, min_chunk_size=500)),
]

chunker_results = {}
for name, chunker in chunker_configs:
    cr = chunker.chunk(ref_result)
    chunker_results[name] = cr
    sizes = [len(c.text) for c in cr.chunks]
    print(f"  {name:<35} chunks={len(cr.chunks):>4}  min={min(sizes):>5}  max={max(sizes):>5}  avg={sum(sizes)//len(sizes):>5}")

### Chunk text previews

In [ ]:
for name, cr in chunker_results.items():
    print(f"\n{'─' * 50}")
    print(f"  {name}")
    print(f"{'─' * 50}")
    for i, c in enumerate(cr.chunks[:2]):
        preview = c.text[:120].replace('\n', ' ')
        print(f"  Chunk {i} [{len(c.text)} chars]: {preview}...")
    if len(cr.chunks) > 2:
        print(f"  ... and {len(cr.chunks) - 2} more chunks")

---
## 4 — Embedders

Run every embedder configuration on a shared chunking result.

In [ ]:
from ragbandit.documents import (
    MistralEmbedder,
    OpenAIEmbedder,
    VoyageAIEmbedder,
    CohereEmbedder,
)

# Use the first chunker result as shared input
shared_chunks = chunker_results["FixedSize(1000/200)"]

embedder_configs = [
    ("Mistral (mistral-embed)",           MistralEmbedder(api_key=MISTRAL_API_KEY, model="mistral-embed")),
    ("OpenAI (text-embedding-3-small)",   OpenAIEmbedder(api_key=OPENAI_API_KEY, model="text-embedding-3-small")),
    ("OpenAI (text-embedding-3-large)",   OpenAIEmbedder(api_key=OPENAI_API_KEY, model="text-embedding-3-large")),
    ("VoyageAI (voyage-3)",               VoyageAIEmbedder(api_key=VOYAGE_API_KEY, model="voyage-3")),
    ("VoyageAI (voyage-3-large)",         VoyageAIEmbedder(api_key=VOYAGE_API_KEY, model="voyage-3-large")),
    ("Cohere (embed-v4.0)",               CohereEmbedder(api_key=COHERE_API_KEY, model="embed-v4.0")),
    ("Cohere (embed-english-v3.0)",       CohereEmbedder(api_key=COHERE_API_KEY, model="embed-english-v3.0")),
]

for name, embedder in embedder_configs:
    print(f"\n{'─' * 50}")
    print(f"  {name}")
    print(f"{'─' * 50}")
    er = embedder.embed_chunks(shared_chunks)
    dim = len(er.chunks_with_embeddings[0].embedding) if er.chunks_with_embeddings else 0
    print(f"  Vectors    : {len(er.chunks_with_embeddings)}")
    print(f"  Dimensions : {dim}")
    print(f"  Model      : {er.model_name}")
    if er.chunks_with_embeddings:
        print(f"  Sample     : {er.chunks_with_embeddings[0].embedding[:5]}")